# Singapore HDB Resale Price Prediction

This notebook is **Part 2** of the Singapore HDB resale project.  
Part 1 focused on **visualization, mapping, and exploratory analysis**.  
This notebook shifts to **predictive modeling**: can historical resale transactions be used to estimate what a flat would sell for?

## Main goal
Build and compare multiple supervised machine-learning models that predict `resale_price` from:

- flat size and configuration
- age / lease-related variables
- storey information
- sale timing
- geospatial location
- neighborhood / street context

## Practical motivation
The most practical question behind this notebook is simple:

> **If I wanted to sell my own HDB flat today, can I use past market transactions to estimate a reasonable market price?**

That makes this notebook more than just a machine-learning exercise. It is also a valuation-oriented workflow:
1. prepare a clean transaction dataset
2. engineer economically meaningful features
3. compare model families fairly
4. interpret what the strongest model has learned
5. apply the model to a real flat as a pricing aid

## Learning roadmap
The notebook is organized as an end-to-end modeling workflow:

1. import and inspect the prepared transaction data
2. engineer predictive features from raw columns
3. split the data into train / validation / test sets
4. define reusable preprocessing and evaluation helpers
5. train several model families
6. compare out-of-sample performance
7. visualize actual vs predicted values
8. interpret the best tree model with feature importance and SHAP
9. use dimension reduction for intuition-building
10. summarize what the results imply for practical pricing

## Why compare multiple models?
Different algorithms capture different kinds of patterns:

- **linear models** provide an interpretable baseline
- **polynomial regression** allows limited curvature while staying relatively simple
- **tree ensembles** capture non-linear interactions that are common in housing data
- **k-NN** acts like an automated “similar recent sales” benchmark
- **neural networks** offer another flexible non-linear function class

The point is not just to fit one fancy model.  
It is to learn **which modeling assumptions work best for HDB resale prices** and whether the performance is strong enough to be useful in practice.

In [ ]:

# Optional installs (uncomment only if needed in your environment)
# %pip install tqdm umap-learn shap xgboost lightgbm


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import shap
import umap

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.rcParams["axes.grid"] = True

## 1. Import data

This section loads the cleaned resale transaction file together with the geocoded location information created in the visualization notebook.

## Why this notebook depends on Part 1
A major theme of the project is that **location is one of the strongest drivers of housing price**.  
Without geocodes, the model would only know broad administrative labels such as town or street name.  
With latitude and longitude, it can learn much richer spatial price patterns.

That is why the recommended run order is:

1. `singapore_hdb_resale_visualization.ipynb`
2. `singapore_hdb_resale_prediction.ipynb`

## What to check in this section
Before any modeling starts, it is important to verify that:

- the data loaded correctly
- the row count looks reasonable
- numeric columns have plausible ranges
- categorical columns contain the expected values
- resale price has a distribution that matches intuition

This first inspection step helps catch data-quality problems early, before they silently affect model performance later.

### Import CSV data

The raw transaction CSV is the main modeling table.

The first few cells deliberately do only **foundational cleaning**:
- convert important fields into numeric form
- standardize the key transaction columns
- remove clearly unusable rows
- keep the table stable before adding more complex features

This is good workflow design.  
It separates **basic data integrity work** from **feature engineering**, so a future reader can understand which columns came directly from the source and which were derived later.

In [ ]:
# Update this path if your CSV is saved somewhere else.
DATA_PATH = Path("outputs/transactions_clean.csv")
df = pd.read_csv(DATA_PATH)

In [ ]:
# Numeric summary
display(df.describe(include=[np.number]).T)

# Categorical summary
categorical_cols_raw = df.select_dtypes(include="object").columns.tolist()
display(df[categorical_cols_raw].describe().T)

In [ ]:
target_summary = pd.Series({
    "rows": len(df),
    "min_price": df["resale_price"].min(),
    "median_price": df["resale_price"].median(),
    "mean_price": df["resale_price"].mean(),
    "max_price": df["resale_price"].max(),
    "std_price": df["resale_price"].std(),
})
display(target_summary)

plt.figure(figsize=(5, 4))
plt.hist(df["resale_price"], bins=60)
plt.title("Distribution of Resale Price")
plt.xlabel("Resale Price")
plt.ylabel("Count")
plt.show()


## 2. Feature engineering

Raw administrative datasets rarely arrive in the exact form that a predictive model needs.  
This section turns the transaction table into a richer modeling dataset.

## Purpose of feature engineering here
The goal is to create variables that reflect the economics of HDB pricing more directly, including:

- **time of sale**: captures overall market movement
- **property age / lease balance**: approximates remaining economic value
- **storey level**: captures vertical premium effects
- **flat size**: one of the strongest direct drivers of price
- **location**: captures spatial differences in demand
- **price per square meter (`psm`)**: useful for exploratory benchmarking

## Why feature engineering matters so much
Machine-learning models do not understand housing logic on their own.  
They only see the variables we choose to provide.

For property data, thoughtful feature engineering often matters just as much as model choice because it:
- converts text fields into learnable quantities
- makes hidden structure explicit
- gives simpler models a better chance to perform well
- improves interpretability later when explaining model outputs

### Function rationale: translating raw text fields into numeric features

Several useful HDB variables are stored as text rather than as ready-to-model numbers.  
Examples include:

- `remaining_lease`
- `storey_range`

Models generally work better when those columns are converted into numeric quantities.  
So the next functions extract information such as:

- **remaining lease in months**
- **midpoint of the storey range**

This is a classic but important modeling step:  
turning human-readable administrative labels into machine-readable signals without losing too much meaning.

In [ ]:
def parse_remaining_lease_to_months(text):
    """Convert lease text into a single numeric value measured in months."""
    if pd.isna(text):
        return np.nan

    text = str(text).strip().lower()
    years = 0
    months = 0

    import re
    y = re.search(r"(\d+)\s+year", text)
    m = re.search(r"(\d+)\s+month", text)

    if y:
        years = int(y.group(1))
    if m:
        months = int(m.group(1))

    return years * 12 + months

def parse_storey_mid(text):
    """Approximate a storey-range string by its midpoint for numeric modeling."""
    if pd.isna(text):
        return np.nan
    try:
        low, _, high = str(text).partition(" TO ")
        return (int(low) + int(high)) / 2
    except Exception:
        return np.nan

model_df_full = df.copy()

# time features
model_df_full["month"] = pd.to_datetime(model_df_full["month"], format="%Y-%m-%d")
model_df_full["sale_year"] = model_df_full["month"].dt.year
model_df_full["sale_month_num"] = model_df_full["month"].dt.month
# model_df_full["sale_quarter"] = model_df_full["month"].dt.quarter

# lease / age features
model_df_full["remaining_lease_months"] = model_df_full["remaining_lease"].apply(parse_remaining_lease_to_months)
model_df_full["property_age_at_sale"] = model_df_full["sale_year"] - model_df_full["lease_commence_date"]

# floor / area related
model_df_full["storey"] = model_df_full["storey_range"].apply(parse_storey_mid)

# simplified location key
model_df_full["address"] = model_df_full["block"].astype(str) + " " + model_df_full["street_name"].astype(str)

# log target for optional analysis if needed
model_df_full["log_resale_price"] = np.log1p(model_df_full["resale_price"])

model_df_full.head()

In [ ]:
engineered_numeric = [
    "floor_area_sqm",
    # "lease_commence_date",
    # "remaining_lease_months",
    "property_age_at_sale",
    "storey",
    "sale_year",
    "sale_month_num",
    # "sale_quarter",
    "latitude",
    "longitude",
]

engineered_categorical = [
    "town",
    "flat_type",
    "flat_model",
    "street_name",
    # "storey_range",
]

keep_cols = engineered_numeric + engineered_categorical + ["resale_price", "psm", "log_resale_price"]
model_df = model_df_full[keep_cols].copy()

print(model_df.shape)
model_df.head()


In [ ]:
corr1 = model_df.select_dtypes(include=(['int32', 'int64', 'float64'])).corr('spearman')['psm'].drop(["psm", "resale_price", "log_resale_price"]).sort_values(ascending=True)
corr2 = model_df.select_dtypes(include=(['int32', 'int64', 'float64'])).corr('spearman')['resale_price'].drop(["psm", "resale_price", "log_resale_price"]).sort_values(ascending=True)

f, (ax1, ax2) =  plt.subplots(nrows=1, ncols=2, figsize=(10,4))

ax1.barh(corr1.index, corr1.values)
ax1.set_xlabel("Correlation with Price per sqm")
ax1.set_title("Feature correlation with \n Price per sqm")

ax2.barh(corr2.index, corr2.values)
ax2.set_xlabel("Correlation with Resale Price")
ax2.set_title("Feature correlation with \n Resale Price")

plt.tight_layout()
plt.show()


In [ ]:
corr3 = model_df.select_dtypes(include=(['int32', 'int64', 'float64'])).corr('spearman')
sns.heatmap(corr3, cmap='RdBu_r')

## 3. Train, Validate, Test Data

This section assigns different subsets of the data to different modeling roles.

## Why use three splits instead of one?
- **training set**: used to fit model parameters
- **validation set**: used to compare hyperparameters and model families
- **test set**: used only for the final unbiased evaluation

That separation is critical.  
If the test set is repeatedly used during tuning, model selection can gradually overfit to the evaluation data, giving overly optimistic results.

## Practical interpretation
Because the purpose of this notebook is valuation-like prediction, the held-out test set is especially important.  
It is the best approximation in the notebook to the real-world question:

> *How well would the model perform on transactions it has never seen before?*

In [ ]:
X = model_df.drop(columns=["resale_price", "psm", "log_resale_price"])
y = model_df["resale_price"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, random_state=RANDOM_STATE
)
# This gives about 70% train / 15% val / 15% test overall.

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


## 4. Helper functions and preprocessors

The next section defines reusable utilities for metrics, tuning, fitting, and preprocessing.

## Why centralize this logic?
Without helper functions, every model section would need to repeat the same boilerplate for:
- fitting
- prediction
- metric calculation
- timing
- validation tuning
- result storage

Centralizing those steps makes the notebook:

- easier to read
- easier to debug
- easier to extend with additional models later
- more consistent across experiments

That consistency matters because fair model comparison requires the same evaluation workflow across all candidate methods.

### Function rationale: common metrics and reusable evaluation helpers

The helper functions below create the evaluation backbone of the notebook.

## Why use multiple regression metrics?
No single number tells the whole story:

- **RMSE** penalizes large misses more strongly
- **MAE** captures the average absolute error in dollar terms
- **R²** measures how much variation the model explains
- **MAPE** expresses error as a percentage, which is often intuitive for pricing tasks

This combination is useful because a model intended for resale-price estimation should be judged from several angles, not only from one score.

## Why store fitted models and predictions?
Later sections need to:
- compare models side by side
- draw actual-vs-predicted plots
- interpret the strongest tree-based model
- reuse the winning model for a real flat example

Keeping those objects in shared dictionaries makes the later notebook sections much more transparent and reusable.

In [ ]:
results = []
fitted_models = {}
predictions = {}

def regression_metrics(y_true, y_pred):
    """Compute a small set of complementary regression metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    nonzero_mask = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "MAPE_pct": mape,
    }

def add_result(model_name, fitted_model, X_eval, y_eval, fit_seconds=None, notes=None):
    """Evaluate a fitted model and store both metrics and predictions for later use."""
    pred = fitted_model.predict(X_eval)
    metrics = regression_metrics(y_eval, pred)
    metrics["Model"] = model_name
    if fit_seconds is not None:
        metrics["Fit_seconds"] = fit_seconds
    if notes is not None:
        metrics["Notes"] = notes
    results.append(metrics)
    fitted_models[model_name] = fitted_model
    predictions[model_name] = pred
    return pred, metrics

def sample_rows(X_df, y_series, max_rows, random_state=RANDOM_STATE):
    """Randomly subsample large datasets to keep expensive models practical."""
    if (max_rows is None) or (len(X_df) <= max_rows):
        return X_df.copy(), y_series.copy()
    sampled_index = X_df.sample(n=max_rows, random_state=random_state).index
    return X_df.loc[sampled_index].copy(), y_series.loc[sampled_index].copy()

def evaluate_on_validation(pipeline, X_tr, y_tr, X_va, y_va):
    """Fit one pipeline on the training split and score it on the validation split."""
    pipeline.fit(X_tr, y_tr)
    pred = pipeline.predict(X_va)
    return regression_metrics(y_va, pred)

def tune_with_progress(
    model_name,
    pipeline_builder,
    param_distributions,
    X_tr,
    y_tr,
    X_va,
    y_va,
    n_iter=8,
    sample_limit=None,
    random_state=RANDOM_STATE
):
    """Sample hyperparameters, evaluate them on the validation set, and keep the best one."""
    X_tr_use, y_tr_use = X_tr, y_tr
    if sample_limit is not None:
        X_tr_use, y_tr_use = sample_rows(X_tr, y_tr, sample_limit, random_state=random_state)

    sampled_params = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=random_state))

    tuning_rows = []
    best_score = np.inf
    best_params = None
    best_pipeline = None

    print(f"Tuning {model_name} on {len(X_tr_use):,} training rows and {len(X_va):,} validation rows")

    for params in tqdm(sampled_params, desc=f"{model_name} tuning"):
        pipeline = pipeline_builder(**params)
        start = time.time()
        metrics = evaluate_on_validation(pipeline, X_tr_use, y_tr_use, X_va, y_va)
        elapsed = time.time() - start

        row = {"fit_seconds": elapsed, **params, **metrics}
        tuning_rows.append(row)

        if metrics["RMSE"] < best_score:
            best_score = metrics["RMSE"]
            best_params = params
            best_pipeline = pipeline

    tuning_df = pd.DataFrame(tuning_rows).sort_values("RMSE").reset_index(drop=True)
    return best_params, tuning_df, best_pipeline

def fit_model_on_sample(pipeline, X_train_data, y_train_data, sample_limit=None, desc=None):
    """Fit a pipeline on all rows or on a controlled sample and return runtime metadata."""
    X_fit, y_fit = sample_rows(X_train_data, y_train_data, sample_limit, random_state=RANDOM_STATE)
    if desc is not None:
        print(f"{desc}: fitting on {len(X_fit):,} rows")
    start = time.time()
    pipeline.fit(X_fit, y_fit)
    fit_seconds = time.time() - start
    return pipeline, fit_seconds, len(X_fit)

### Preprocessing rationale: why there are two preprocessors

The notebook defines two preprocessing pipelines because different model families have different numerical needs.

- **general preprocessor**  
  imputes missing values, scales numeric inputs, and one-hot encodes categoricals  
  → useful for linear models, k-NN, and neural networks

- **tree preprocessor**  
  imputes missing values and one-hot encodes categoricals, but does **not** scale numerics  
  → appropriate for Random Forest, XGBoost, and LightGBM

## Why this matters
Scaling is essential for distance-based and gradient-based models, but it is usually unnecessary for tree models.  
Using separate preprocessors keeps each model family in a setting that suits its learning mechanics.

This is a subtle but important sign of good ML workflow:  
the notebook is not forcing every algorithm through the exact same data treatment when that would be suboptimal.

In [ ]:
# General preprocessing
numeric_features = engineered_numeric
categorical_features = engineered_categorical

general_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)

# Tree-friendly preprocessing: impute numerics, one-hot categoricals, no numeric scaling needed
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)


## 5. Linear Regression

Linear regression is the baseline model.

## Why start here?
It is simple, fast, and interpretable.  
It answers the question:

> *How far can we get if resale price is modeled as an approximately linear combination of the available features?*

That makes it a useful benchmark.  
More complex models should earn their complexity by meaningfully improving on this baseline.

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", general_preprocessor),
    ("model", LinearRegression())
])

start = time.time()
linear_model.fit(X_train_full, y_train_full)
linear_fit_seconds = time.time() - start

linear_pred, linear_metrics = add_result(
    "Linear Regression",
    linear_model,
    X_test,
    y_test,
    fit_seconds=linear_fit_seconds
)

linear_metrics

## 6. Polynomial Regression

Polynomial regression extends the linear baseline by allowing selected numeric relationships to bend rather than remain perfectly straight.

## Why try this?
HDB prices are unlikely to be strictly linear in variables such as:
- floor area
- age
- timing
- location-related effects

Polynomial features let the model capture some curvature while still staying within a fairly interpretable regression framework.

## What this comparison tells us
If polynomial regression improves substantially over plain linear regression, that is early evidence that the pricing surface is meaningfully non-linear even before moving to tree ensembles.

In [ ]:
poly_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)

poly_model = Pipeline(steps=[
    ("preprocessor", poly_preprocessor),
    ("model", LinearRegression())
])

start = time.time()
poly_model.fit(X_train_full, y_train_full)
poly_fit_seconds = time.time() - start

poly_pred, poly_metrics = add_result(
    "Polynomial Regression",
    poly_model,
    X_test,
    y_test,
    fit_seconds=poly_fit_seconds
)

poly_metrics

## 7. Random Forest

Random Forest averages predictions from many decision trees.

## Why it is a sensible candidate here
Housing prices often involve:
- non-linear relationships
- threshold effects
- interactions between size, age, town, and location

Random Forest can capture those patterns without requiring heavy manual feature transformation.  
It is also robust on mixed tabular data and provides feature-importance outputs that are useful later for interpretation.

### Model-building rationale: Random Forest

The next function builds a complete **pipeline**, not just a standalone model object.

That is important because preprocessing and prediction then stay bundled together:
- training data are transformed consistently
- validation and test data receive the exact same treatment
- inference later uses the same preprocessing rules as training
- tuning becomes much safer and easier to reproduce

For a notebook that aims to support practical price estimation, this end-to-end pipeline structure is much better than manually transforming data in separate steps.

In [ ]:
# Construct a full preprocessing-plus-model pipeline for random forest regression.
def build_rf_pipeline(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
):
    return Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=True,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

rf_param_distributions = {
    "n_estimators": [60, 100, 140],
    "max_depth": [14, 20, 28],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5],
}

rf_best_params, rf_tuning_df, _ = tune_with_progress(
    model_name="Random Forest",
    pipeline_builder=build_rf_pipeline,
    param_distributions=rf_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=6,
    sample_limit=30000
)

display(rf_tuning_df.head(10))

print("Best RF params:", rf_best_params)

best_rf_model = build_rf_pipeline(**rf_best_params)

RF_FINAL_SAMPLE = 60000
best_rf_model, rf_fit_seconds, rf_rows_used = fit_model_on_sample(
    best_rf_model,
    X_train_full,
    y_train_full,
    sample_limit=RF_FINAL_SAMPLE,
    desc="Random Forest"
)

rf_pred, rf_metrics = add_result(
    "Random Forest",
    best_rf_model,
    X_test,
    y_test,
    fit_seconds=rf_fit_seconds,
    notes=f"trained on {rf_rows_used:,} sampled rows"
)

rf_metrics

## 8. k-Nearest Neighbors

k-NN predicts a price by looking at similar historical observations in feature space.

## Why include it?
Conceptually, it resembles a basic valuation instinct:

> *What did similar flats sell for?*

That makes it a nice benchmark for resale pricing.  
If similarity is defined well enough, k-NN can perform surprisingly well on structured tabular data.

## Limitation to keep in mind
k-NN is very sensitive to:
- feature scaling
- irrelevant dimensions
- the definition of “nearby” in a high-dimensional space

So its performance is informative not only about the model itself, but also about how well the engineered features capture practical comparability between flats.

In [ ]:
# Construct a preprocessing-plus-model pipeline for k-nearest-neighbors regression.
def build_knn_pipeline(
    n_neighbors=20,
    weights="distance",
    p=2,
):
    return Pipeline(steps=[
        ("preprocessor", general_preprocessor),
        ("model", KNeighborsRegressor(
            n_neighbors=n_neighbors,
            weights=weights,
            p=p,
            n_jobs=-1
        ))
    ])
    
knn_param_distributions = {
    "n_neighbors": [5, 10, 20, 30, 40],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

knn_best_params, knn_tuning_df, _ = tune_with_progress(
    model_name="KNN",
    pipeline_builder=build_knn_pipeline,
    param_distributions=knn_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=8,
    sample_limit=40000
)

display(knn_tuning_df.head(10))

best_knn_model = build_knn_pipeline(**knn_best_params)

start = time.time()
best_knn_model.fit(X_train_full, y_train_full)
knn_fit_seconds = time.time() - start

knn_pred, knn_metrics = add_result(
    "k-Nearest Neighbors",
    best_knn_model,
    X_test,
    y_test,
    fit_seconds=knn_fit_seconds
)

knn_metrics

## 9. Neural Network (MLPRegressor)

This section tests a multilayer perceptron, a simple feed-forward neural network for regression.

## Why test a neural network on tabular housing data?
A neural network offers another flexible non-linear function approximator that is very different from tree ensembles.  
It may detect patterns that simpler linear models miss.

At the same time, tabular datasets do not always favor neural nets over boosted trees.  
So this comparison is useful in a practical sense:
- if the neural net wins, that suggests strong smooth non-linear structure
- if it does not, that reinforces how strong tree boosting often is on structured property data

In [ ]:
# Construct a preprocessing-plus-model pipeline for a feed-forward neural network regressor.
def build_mlp_pipeline(
    hidden_layer_sizes=(64, 32),
    alpha=0.001,
    learning_rate_init=0.003,
):
    return Pipeline(steps=[
        ("preprocessor", general_preprocessor),
        ("model", MLPRegressor(
            hidden_layer_sizes=hidden_layer_sizes,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=120,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            random_state=RANDOM_STATE
        ))
    ])
    
mlp_param_distributions = {
    "hidden_layer_sizes": [(32,), (64,), (64, 32)],
    "alpha": [1e-4, 1e-3, 1e-2],
    "learning_rate_init": [0.001, 0.003]
}

mlp_best_params, mlp_tuning_df, _ = tune_with_progress(
    model_name="MLP",
    pipeline_builder=build_mlp_pipeline,
    param_distributions=mlp_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=5,
    sample_limit=25000
)

display(mlp_tuning_df.head(10))

print("Best MLP params:", mlp_best_params)

best_mlp_model = build_mlp_pipeline(**mlp_best_params)

MLP_FINAL_SAMPLE = 40000
best_mlp_model, mlp_fit_seconds, mlp_rows_used = fit_model_on_sample(
    best_mlp_model,
    X_train_full,
    y_train_full,
    sample_limit=MLP_FINAL_SAMPLE,
    desc="MLP"
)

mlp_pred, mlp_metrics = add_result(
    "Neural Network (MLP)",
    best_mlp_model,
    X_test,
    y_test,
    fit_seconds=mlp_fit_seconds,
    notes=f"trained on {mlp_rows_used:,} sampled rows"
)

mlp_metrics

## 10. XGBoost

XGBoost is a gradient-boosted tree model and is often one of the strongest baseline choices for tabular prediction tasks.

## Why it is promising here
Boosting builds trees sequentially so that each new tree focuses on residual errors left by earlier trees.  
That often produces very strong performance on datasets like housing transactions, where:
- effects are non-linear
- feature interactions matter
- local pattern corrections can improve price accuracy

### Model-building rationale: boosted trees

The next two sections test **XGBoost** and **LightGBM**, which are often among the most competitive methods for structured tabular data.

The tuning grids are intentionally **moderate rather than huge**.  
That reflects a practical trade-off:
- enough variation to compare sensible parameter settings
- not so much that the notebook becomes unnecessarily slow or difficult to reuse

This is especially appropriate for an applied notebook whose goal is insight and practical prediction, not exhaustive benchmark optimization.

In [ ]:
# Construct a preprocessing-plus-model pipeline for XGBoost regression.
def build_xgb_pipeline(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
):
    return Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    
xgb_param_distributions = {
    "n_estimators": [200, 300, 500],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

xgb_best_params, xgb_tuning_df, _ = tune_with_progress(
    model_name="XGBoost",
    pipeline_builder=build_xgb_pipeline,
    param_distributions=xgb_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=8,
    sample_limit=80000
)

display(xgb_tuning_df.head(10))

best_xgb_model = build_xgb_pipeline(**xgb_best_params)

start = time.time()
best_xgb_model.fit(X_train_full, y_train_full)
xgb_fit_seconds = time.time() - start

xgb_pred, xgb_metrics = add_result(
    "XGBoost",
    best_xgb_model,
    X_test,
    y_test,
    fit_seconds=xgb_fit_seconds
)

display(xgb_metrics)

## 11. LightGBM

LightGBM is another gradient-boosted tree framework designed for efficiency and strong performance on large tabular datasets.

## Why compare it with XGBoost?
Both are strong boosted-tree methods, but they differ in:
- tree-growth strategy
- speed / memory behavior
- hyperparameter sensitivity
- sometimes final accuracy

Testing both helps answer a genuinely empirical question:

> *For this HDB resale dataset, which boosting implementation learns the pricing structure more effectively?*

In [ ]:
# Construct a preprocessing-plus-model pipeline for LightGBM regression.
def build_lgbm_pipeline(
    n_estimators=400,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
):
    return Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", LGBMRegressor(
            n_estimators=n_estimators,
            num_leaves=num_leaves,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

lgbm_param_distributions = {
    "n_estimators": [250, 400, 600],
    "num_leaves": [31, 63, 127],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

lgbm_best_params, lgbm_tuning_df, _ = tune_with_progress(
    model_name="LightGBM",
    pipeline_builder=build_lgbm_pipeline,
    param_distributions=lgbm_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=8,
    sample_limit=80000
)

display(lgbm_tuning_df.head(10))

best_lgbm_model = build_lgbm_pipeline(**lgbm_best_params)

start = time.time()
best_lgbm_model.fit(X_train_full, y_train_full)
lgbm_fit_seconds = time.time() - start

lgbm_pred, lgbm_metrics = add_result(
    "LightGBM",
    best_lgbm_model,
    X_test,
    y_test,
    fit_seconds=lgbm_fit_seconds
)

display(lgbm_metrics)

## 12. Model comparison

Once all candidate models are trained, this section compares them on the held-out test set.

## How to read the table
- **lower RMSE / MAE** → smaller dollar errors
- **higher R²** → more variance explained
- **lower MAPE** → smaller average percentage error

## Why this section matters
This is where model selection becomes evidence-based.  
Instead of assuming that a more complex model is better, the notebook asks the data to show which method generalizes best.

In [ ]:
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    ordered_cols = ["Model", "RMSE", "MAE", "R2", "MAPE_pct", "Fit_seconds", "Notes"]
    existing_cols = [col for col in ordered_cols if col in results_df.columns]
    results_df = results_df[existing_cols].sort_values("RMSE").reset_index(drop=True)

display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print("Best model based on lowest test RMSE:", best_model_name)

### Interpreting the comparison results

The comparison table shows a very clear ranking:

- **LightGBM is the strongest overall model**
- **XGBoost is also strong, but not quite as accurate here**
- simpler baselines such as linear and polynomial regression still perform reasonably well
- the best tree-based models materially outperform the simpler models, which suggests that HDB resale pricing is meaningfully **non-linear**

Using the displayed test metrics:

- **LightGBM RMSE ≈ 26.4k SGD**
- **LightGBM MAE ≈ 18.8k SGD**
- **LightGBM R² ≈ 0.981**
- **LightGBM MAPE ≈ 3.67%**

That is a strong result for a broad resale-price model built from transactional, structural, and location features.

## Practical takeaway
For valuation-style use, the most intuitive figures are often **MAE** and **MAPE**:

- an average absolute error of about **18.8k SGD**
- an average percentage error of about **3.7%**

This suggests the best model is accurate enough to be genuinely useful as a **pricing support tool**, while still not being a perfect substitute for a full market appraisal or current listing intelligence.

In [ ]:

plt.figure(figsize=(5, 3))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison by RMSE")
plt.ylabel("RMSE")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
plt.bar(results_df["Model"], results_df["R2"])
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison by R²")
plt.ylabel("R²")
plt.tight_layout()
plt.show()


## 13. Actual vs predicted plots

Metric tables summarize performance, but scatter plots reveal *how* the errors are distributed.

## Why these plots matter
They show whether a model:
- follows the diagonal closely across the full price range
- systematically underpredicts expensive flats
- overpredicts cheaper flats
- becomes noisier in certain parts of the market

For a resale-price model, this matters because average accuracy alone can hide poor behavior at the higher-value end of the market.

In [ ]:
f, ([ax1, ax2, ax3, ax4], [ax5, ax6, ax7, ax8]) =  plt.subplots(2, 4, figsize=(24,12))

for i in range(0,len(results_df["Model"])):
    model_name = results_df["Model"][i]
    axes =[ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8][i]
    
    pred = predictions[model_name]
    axes.scatter(y_test, pred, alpha=0.25, s=10)
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    axes.plot([min_val, max_val], [min_val, max_val], linestyle="--", c="red")
    axes.set_title(f"{model_name}")
    axes.set_xlabel("Actual Price")
    axes.set_ylabel("Predicted Price")

plt.show()


### How well can the model actually predict a real flat?

A very practical test is to feed the model the details of a real flat rather than just rely on aggregate test metrics.

Here the notebook checks a personal HDB transaction:

- **5-room flat**
- **Sengkang (Rivervale Crescent)**
- **purchased in August 2016**
- **actual transaction price: 415,000 SGD**

This is a meaningful sanity check because it asks whether the model gives a sensible answer for a real-world case that matters personally.

## Important caveat
The notebook notes that this sale occurred before the main modeled period begins in January 2017, so this is slightly outside the training time range.  
That makes the exercise more demanding and therefore a stronger real-world sanity check.

In [ ]:
pred = best_lgbm_model.predict(pd.DataFrame([{
    'floor_area_sqm': 110, 
    'property_age_at_sale': 15, 
    'storey': 15, 
    'sale_year': 2016, 
    'sale_month_num': 8, 
    'latitude': 1.38841338,
    'longitude': 103.9066762,
    'town': 'SENGKANG', 
    'flat_type': '5 ROOM', 
    'flat_model': 'Improved', 
    'street_name': 'RIVERVALE CRES'
}]))

print(f'For my own HDB flat (5 Room, Sengkang, bought in Aug 2016):')
print(f"-- The predicted transaction price was: SGD {pred[0]:,.0f}")
print(f"-- My actual transaction price was: SGD 415,000")

The predicted value of **410,164 SGD** is very close to the actual purchase price of **415,000 SGD**.

## Why this is encouraging
The exact match is not the point.  
What matters is that the error is small relative to the transaction value, which suggests that the model has learned a market structure that is realistic enough to handle an individual flat example reasonably well.

That gives confidence to the next step: using the fitted model as a **current pricing guide** for the same flat under updated sale conditions.

In [ ]:
pred = best_lgbm_model.predict(pd.DataFrame([{
    'floor_area_sqm': 110, 
    'property_age_at_sale': 25, 
    'storey': 15, 
    'sale_year': 2026, 
    'sale_month_num': 3, 
    'latitude': 1.38841338,
    'longitude': 103.9066762,
    'town': 'SENGKANG', 
    'flat_type': '5 ROOM', 
    'flat_model': 'Improved', 
    'street_name': 'RIVERVALE CRES'
}]))

print(f'For my own HDB flat, if I were to sell it today:')
print(f"-- The predicted transaction price will be: SGD {pred[0]:,.0f}")


## 14. Feature importance and SHAP

After identifying the strongest tree-based model, the notebook moves from **prediction** to **interpretation**.

## Two complementary questions
1. **Which features matter most overall?**  
   → answered by feature importance

2. **How do those features push specific predictions up or down?**  
   → explored with SHAP

This is an important step because a high-performing model is more useful when we can also explain what it appears to be learning about HDB price formation.

In [ ]:
tree_model_candidates = [name for name in results_df["Model"] if name in ["LightGBM", "XGBoost", "Random Forest"]]
best_tree_model_name = tree_model_candidates[0] if len(tree_model_candidates) > 0 else None
print("Best tree-based model:", best_tree_model_name)

In [ ]:
best_tree_model = fitted_models[best_tree_model_name]
tree_pre = best_tree_model.named_steps["preprocessor"]
tree_est = best_tree_model.named_steps["model"]

feature_names = tree_pre.get_feature_names_out()

if hasattr(tree_est, "feature_importances_"):
    feature_importances = pd.DataFrame({
        "feature": feature_names,
        "importance": tree_est.feature_importances_
    }).sort_values("importance", ascending=False)

    display(feature_importances.head(20))

    top_n = 20
    top_feat = feature_importances.head(top_n).sort_values("importance")
    plt.figure(figsize=(8, 5))
    plt.barh(top_feat["feature"], top_feat["importance"])
    plt.title(f"Top {top_n} Feature Importances: {best_tree_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


### What the feature-importance results suggest

The top-ranked features are dominated by:

- **latitude**
- **floor area**
- **longitude**
- **property age at sale**
- **sale year**
- **storey**
- **sale month**

This is a very plausible result.

## Interpretation
The model appears to be learning three major price drivers:

1. **Location matters enormously**  
   Latitude and longitude ranking near the very top shows that price differences are not explained fully by town labels alone.  
   Fine-grained spatial position contains major valuation signal.

2. **Size and age are core structural drivers**  
   Larger flats tend to command higher prices, while older flats often face lease-related discounts.

3. **Timing matters because the resale market moves over time**  
   Sale year and sale month are useful because the HDB market is not static; prices shift with broader market conditions.

Overall, the feature ranking is economically intuitive, which strengthens confidence that the model is learning meaningful housing patterns rather than only statistical noise.

In [ ]:
best_tree_model = fitted_models[best_tree_model_name]
tree_pre = best_tree_model.named_steps["preprocessor"]
tree_est = best_tree_model.named_steps["model"]

X_shap_sample, y_shap_sample = sample_rows(X_test, y_test, max_rows=1200, random_state=RANDOM_STATE)
X_shap_transformed = tree_pre.transform(X_shap_sample)

if hasattr(X_shap_transformed, "toarray"):
    # Keep sample moderate to avoid excessive memory use
    X_shap_dense = X_shap_transformed.toarray()
else:
    X_shap_dense = X_shap_transformed

try:
    explainer = shap.Explainer(tree_est, X_shap_dense, feature_names=tree_pre.get_feature_names_out())
    shap_values = explainer(X_shap_dense)

    shap.plots.beeswarm(shap_values, max_display=20)
except Exception as e:
    print("SHAP calculation failed for this model/environment.")
    print("Details:", e)


### Why SHAP is especially useful here

Feature importance gives a global ranking, but SHAP adds direction and local detail.

For example, SHAP can help answer questions such as:
- does a larger floor area generally push the prediction upward?
- does older property age tend to pull value downward?
- how strongly does location change the estimate for a specific flat?

That makes SHAP particularly relevant for practical pricing use, because a seller usually wants not just a number, but also a rough explanation of **why** the model arrived there.

## 15. Dimension reduction for visualization

The feature space becomes fairly high-dimensional after preprocessing, especially once categorical variables are one-hot encoded.

## Why include PCA, t-SNE, and UMAP?
These methods reduce that high-dimensional representation down to two dimensions so we can visually inspect broader structure such as:

- clustering of similar flats
- gradients associated with resale price
- separation between more common and more unusual transactions

## What this section is *not*
This is **not** part of the predictive pipeline used for the final price model.  
It is mainly an exploratory / teaching section that helps build intuition about how structured the transformed feature space is.

In [ ]:

sample_n = min(3000, len(model_df))
viz_df = model_df.sample(sample_n, random_state=RANDOM_STATE).copy()

X_viz = viz_df.drop(columns=["resale_price", "psm", "log_resale_price"])
y_viz = viz_df["resale_price"]

viz_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)

X_viz_transformed = viz_preprocessor.fit_transform(X_viz)

if hasattr(X_viz_transformed, "toarray"):
    X_viz_dense = X_viz_transformed.toarray()
else:
    X_viz_dense = X_viz_transformed

print("Visualization sample size:", len(viz_df))
print("Transformed visualization matrix shape:", X_viz_dense.shape)


In [ ]:
# PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_viz_dense)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_viz, s=12)
plt.colorbar(sc, label="Resale Price")
plt.title("PCA of HDB Resale Data")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)


In [ ]:
# t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE
)
X_tsne = tsne.fit_transform(X_viz_dense)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_viz, s=12)
plt.colorbar(sc, label="Resale Price")
plt.title("t-SNE of HDB Resale Data")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()


In [ ]:
# UMAP
reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
X_umap = reducer.fit_transform(X_viz_dense)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_viz, s=12)
plt.colorbar(sc, label="Resale Price")
plt.title("UMAP of HDB Resale Data")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

### Reading the low-dimensional maps carefully

If similar colors and nearby points tend to cluster together, that suggests the engineered features are capturing meaningful market structure.  
In other words, flats with similar transformed profiles often do sit near one another in the learned representation.

At the same time, these plots should be interpreted cautiously:

- **PCA** preserves broad linear variation
- **t-SNE** emphasizes local neighborhoods
- **UMAP** often balances local and larger-scale structure

So the plots are useful for intuition, but they are not literal maps of Singapore and they should not be treated as causal evidence by themselves.

## 16. Final summary

This notebook shows that HDB resale pricing can be modeled quite effectively using a combination of structural, temporal, and geospatial features.

## Main findings
- The dataset is large enough to support meaningful supervised learning.
- Simple baselines work reasonably well, which is a good sign for data quality.
- **Boosted tree models perform best**, with **LightGBM** emerging as the strongest model in this notebook.
- The winning model achieves errors that are small enough to make it genuinely useful as a practical pricing aid.
- The interpretation section shows that **location, size, age, timing, and storey** are all important contributors to price.

## Answer to the core practical question
For the motivating use case — estimating a reasonable selling price for an HDB flat today — the notebook suggests that the answer is **yes**:

> a well-built model can provide a credible market-price estimate.

In the personal flat example, the model:
- reproduced the historical purchase price quite closely
- produced a current indicative selling estimate of roughly **660k SGD**

## Final caution
This should still be treated as an **evidence-based estimate**, not as a guaranteed market value.  
Actual achievable sale price can also depend on factors not fully captured here, such as:
- the exact renovation condition
- interior view / facing
- unusually recent nearby comparables
- changing short-term market sentiment
- negotiation dynamics

So the best use of this notebook is as a **data-driven valuation support tool**: a strong starting point for pricing, to be combined with current market judgment and comparable recent transactions.

In [ ]:
display(results_df)

print("Best model:", best_model_name)
print()
print("Interpretation tips:")
print("- Lower RMSE / MAE is better.")
print("- Higher R² is better.")
print("- Random Forest and MLP in this notebook are intentionally trained on sampled rows to keep runtime practical.")
print("- If you want the absolute best accuracy later, you can increase RF_FINAL_SAMPLE and MLP_FINAL_SAMPLE gradually.")
print("- The geospatial maps use the geocoded latitude / longitude columns you added.")
print("- Tree boosting models often outperform plain linear models on housing data if the package is available.")
print("- KNN can also be slow when predicting on very large sets.")